# Image eval demo

Run this after the README setup steps: install, create tables, and seed the sample images.

You need `OPENAI_API_KEY` in `.env` for the judge.

In [ ]:
import udfs
import pixeltable as pxt

images = pxt.get_table('img_eval/images')
evals = pxt.get_table('img_eval/evals')

images.where(images.batch_id == 'demo_v1').select(
    images.case_id, images.workflow, images.eval_ready
).collect()

## Grade the sample images

Setting `eval_ready=True` puts those rows on the `evals` view and runs the vision judge. This calls OpenAI and can take a minute.

In [ ]:
images.update({'eval_ready': True}, where=images.batch_id == 'demo_v1')

evals.where(evals.batch_id == 'demo_v1').select(
    evals.case_id, evals.verdict, evals.tags
).collect()

## Look at an image and its scores

Pick one `case_id` from the table above.

In [ ]:
from IPython.display import display

row = (
    evals.where(evals.case_id == 'ui_checkout_mockup')
    .select(evals.image, evals.verdict, evals.scores, evals.tags)
    .collect()[0]
)
display(row['image'])
row['verdict'], row['tags'], row['scores']

## Re-grade the same samples

Scores are cached. Setting `eval_ready` again does not re-run the judge. Recompute `scores` when you want a fresh grade.

In [ ]:
evals.recompute_columns(
    columns=['scores'],
    where=evals.batch_id == 'demo_v1',
)

evals.where(evals.batch_id == 'demo_v1').select(
    evals.case_id, evals.verdict, evals.tags
).collect()

## Generate a new image, then grade it

Leave out a pre-made seed image and the image model generates one on insert. This calls the OpenAI image API.

In [ ]:
images.insert([{
    'case_id': 'my_checkout_v2',
    'batch_id': 'run_2',
    'workflow': 'ui_mockup',
    'task_type': 'image_generation',
    'gen_prompt': (
        'Generate a high-fidelity mobile checkout screen for an ecommerce app. '
        'Orientation: portrait. Include labels Checkout, Place Order, Edit Cart.'
    ),
    'prompt': (
        'Generate a high-fidelity mobile checkout screen for an ecommerce app. '
        'Orientation: portrait. Include labels Checkout, Place Order, Edit Cart.'
    ),
    'criteria': 'Mobile checkout; required labels present; clear CTA.',
    'model': 'gpt-image-1.5',
    'size': '1024x1024',
    'judge_model': 'gpt-5.2',
    'eval_ready': False,
}])

images.update({'eval_ready': True}, where=images.batch_id == 'run_2')

evals.where(evals.batch_id == 'run_2').select(
    evals.case_id, evals.verdict, evals.scores
).collect()

## Edit an image

Needs `GEMINI_API_KEY` in `.env`. These call Gemini on insert (no `seed_image`), then grade.

In [ ]:
from pathlib import Path

logo_dir = Path('use_cases/logo_edit')

images.insert([{
    'case_id': 'logo_year_edit_live',
    'batch_id': 'run_edit_logo',
    'workflow': 'logo_edit',
    'task_type': 'image_editing',
    'gen_prompt': (
        'Edit the logo by changing the text from FIELD to BUTTER .\n'
        'Do not change any other text, colors, shapes, or layout.'
    ),
    'prompt': (
        'Edit the logo by changing the text from FIELD to BUTTER .\n'
        'Do not change any other text, colors, shapes, or layout.'
    ),
    'criteria': (
        'The requested edit is applied exactly.\n'
        'All non-target content remains unchanged.\n'
        'Character style, color, and geometry remain consistent with the original.'
    ),
    'edit_model': 'gemini-2.5-flash-image',
    'judge_model': 'gpt-5.2',
    'ref_image': str(logo_dir / 'assets/logo_input.png'),
    'eval_ready': False,
}])

images.update({'eval_ready': True}, where=images.batch_id == 'run_edit_logo')

evals.where(evals.batch_id == 'run_edit_logo').select(
    evals.case_id, evals.verdict, evals.tags
).collect()


In [ ]:
vto_dir = Path('use_cases/virtual_try_on')

images.insert([{
    'case_id': 'vto_jacket_tryon_live',
    'batch_id': 'run_edit_vto',
    'workflow': 'virtual_try_on',
    'task_type': 'image_editing',
    'gen_prompt': (
        'Put the person in the first image into the jacket shown in the second image.\n'
        "Keep the person's face, pose, body shape, and background unchanged.\n"
        "Preserve the garment's color, pattern, and key details.\n"
        'Do not add extra accessories, text, or new elements.'
    ),
    'prompt': (
        'Put the person in the first image into the jacket shown in the second image.\n'
        "Keep the person's face, pose, body shape, and background unchanged.\n"
        "Preserve the garment's color, pattern, and key details.\n"
        'Do not add extra accessories, text, or new elements.'
    ),
    'criteria': (
        'The output preserves the same person and background.\n'
        'The jacket matches the reference garment closely.\n'
        'Body shape and pose remain consistent outside normal garment effects.\n'
        'The result looks physically plausible.'
    ),
    'edit_model': 'gemini-2.5-flash-image',
    'judge_model': 'gpt-5.2',
    'ref_image': str(vto_dir / 'assets/person.png'),
    'ref_image_2': str(vto_dir / 'assets/garment.png'),
    'eval_ready': False,
}])

images.update({'eval_ready': True}, where=images.batch_id == 'run_edit_vto')

evals.where(evals.batch_id == 'run_edit_vto').select(
    evals.case_id, evals.verdict, evals.tags
).collect()


## Reset the sample batch

Delete the demo rows, then re-run `uv run python scripts/seed_demo.py` from the terminal and grade again.

In [ ]:
images.delete(where=images.batch_id == 'demo_v1')